# Zero-layer transformer

In a zero-layer transformer, the logits are found by $W_U W_E x$ for one-hot $x \in \mathbb R^{d_\text{vocab}}$, $W_E \in \mathbb R^{d_\text{hidden} \times d_\text{vocab}}$, $W_U \in \mathbb R^{d_\text{vocab} \times d_\text{hidden}}$. We claim in this case the model learns, at convergence, the bigram statistics of the training data.

First, we load the corpus; for this demo, we're using Alice in Wonderland, and we use a word-level tokenizer.

In [45]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from src.data import load_corpus

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

corpus = load_corpus()
print(f"{len(corpus):,} tokens, vocab {corpus.vocab_size:,}, device {device}")

34,354 tokens, vocab 2,687, device cuda


Since we use a word-level tokenizer, we can retrieve the bigram statistics simply by checking the `[t, t+1]` pairs.

In [46]:
ids = corpus.tensor(device)
x, y = ids[:-1], ids[1:]

# pairwise_counts[out, in] = how often token `out` directly follows token `in`
pairwise_counts = torch.zeros(corpus.vocab_size, corpus.vocab_size, device=device)
pairwise_counts.index_put_((y, x), torch.ones(len(x), device=device), accumulate=True)

# normalize each column into P(next | current)
bigram_frequencies = pairwise_counts / pairwise_counts.sum(dim=0, keepdim=True)
print(bigram_frequencies)

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 1.0000, 0.0000],
        [0.0140, 0.0000, 0.0188,  ..., 0.0000, 0.0000, 0.0000],
        [0.0820, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
       device='cuda:0')


Now we can write our demo 0-layer transformer, which will be trained on the tokenized corpus and re-assessed.

In [47]:
class ZeroLayerTransformer(nn.Module):
    def __init__(self, vocab_dim: int, h_dim: int, device: str, init_scale=0.02):
        super().__init__()
        self.W_E = nn.Parameter(torch.randn(h_dim, vocab_dim, device=device) * init_scale)
        self.W_U = nn.Parameter(torch.randn(vocab_dim, h_dim, device=device) * init_scale)

    def forward(self, x):
        # logits shape (V, N)
        return self.W_U @ self.W_E[:, x] # pick out the x-th column of E

H_DIM = 128
TRAIN_STEPS = 5000

toy_transformer = ZeroLayerTransformer(vocab_dim=corpus.vocab_size, h_dim=H_DIM, device=device)
opt = torch.optim.Adam(toy_transformer.parameters(), lr=1e-2)
ids = corpus.tensor(device)

x, y = ids[:-1], ids[1:]

losses=[]
for _ in tqdm(range(TRAIN_STEPS), desc="training..."):
    logits = toy_transformer(x)

    # F.cross_entropy expects logits of shape (N, V), but paper has convention (V, N)
    loss = F.cross_entropy(logits.T, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
print(f"Last 5 losses: {losses[-5:]}")

training...: 100%|██████████| 5000/5000 [00:35<00:00, 140.58it/s]

Last 5 losses: [2.9883861541748047, 2.9883103370666504, 2.9882287979125977, 2.9881579875946045, 2.988107204437256]


Note that these last five losses are super close to one another. We argue that this lower bound on the loss arises because the model reaches the limit imposed by the entropy of the natural bigram distribution inside our text. To verify this, we can calculate that entropy:

In [48]:
p_joint = pairwise_counts / pairwise_counts.sum()
seen = pairwise_counts > 0
entropy = (- p_joint[seen] * bigram_frequencies[seen].log()).sum().item()
final_loss = losses[-1]

print(f"Absolute diff between distribution entropy and final loss:\n{abs(entropy - final_loss)}")

Absolute diff between distribution entropy and final loss:
0.0001518726348876953


So we've verified that we have reached the theoretical ceiling of performance for this corpus. Next we recover the log bigram statistics from the learned weight matrix $W_UW_E$, and compare that to the empirical bigram frequencies:

In [49]:
W_E = toy_transformer.W_E.detach()
W_U = toy_transformer.W_U.detach()

# get table, replace unseen stuff with -inf
recovered_frequency_table = F.log_softmax(W_U@W_E, dim=0)
recovered_frequency_table = recovered_frequency_table.where(seen, float('-inf'))
log_frequencies = torch.log(bigram_frequencies)
print(f"Recovered Table:\n{recovered_frequency_table}\nLog Table:{log_frequencies}")

diff = (recovered_frequency_table - log_frequencies)[seen].abs().mean().item()
print(f"Mean difference between recovered matrix and bigram statistic: [{diff}]")

Recovered Table:
tensor([[       -inf,        -inf,        -inf,  ...,        -inf,
         -9.5367e-07,        -inf],
        [-4.2616e+00,        -inf, -3.9612e+00,  ...,        -inf,
                -inf,        -inf],
        [-2.5127e+00,        -inf,        -inf,  ...,        -inf,
                -inf,        -inf],
        ...,
        [       -inf,        -inf,        -inf,  ...,        -inf,
                -inf,        -inf],
        [       -inf,        -inf,        -inf,  ...,        -inf,
                -inf,        -inf],
        [       -inf,        -inf,        -inf,  ...,        -inf,
                -inf,        -inf]], device='cuda:0')
Log Table:tensor([[   -inf,    -inf,    -inf,  ...,    -inf,  0.0000,    -inf],
        [-4.2681,    -inf, -3.9739,  ...,    -inf,    -inf,    -inf],
        [-2.5011,    -inf,    -inf,  ...,    -inf,    -inf,    -inf],
        ...,
        [   -inf,    -inf,    -inf,  ...,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -

So we've done it! We successfully recovered the bigram statistics matrix from training with Adam on a 0-layer transformer.